Langfuse 실습

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "Mon" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "W4" / "Mon"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

프로젝트 루트  : d:\hanwha-agent


In [6]:
import sys
from datetime import datetime, timezone
from pathlib import Path

import anthropic  # noqa: E402
from langfuse import Langfuse  # noqa: E402

from app.core.config import get_settings  # noqa: E402

QUESTION = "부산 출장 숙박비 한도가 얼마인가요?"
SYSTEM = "너는 사내 규정 질의응답 도우미다. 근거가 없으면 없다고 말한다."

settings = get_settings()

if not settings.langfuse_public_key or settings.langfuse_secret_key is None:
    print(".env 의 LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY 가 비어 있습니다.")
    print("키를 붙여 넣고 저장했는지, 레포 루트에서 실행했는지 확인하세요.")
    sys.exit(1)

lf = Langfuse(
    public_key=settings.langfuse_public_key,
    secret_key=settings.langfuse_secret_key.get_secret_value(),
    host=settings.langfuse_host,
)
print("보낼 곳 :", settings.langfuse_host)

# 
try:
    lf.auth_check()
except Exception as e: 
    print("인증 실패 :", type(e).__name__)
    print("  ① 키를 발급한 리전과 LANGFUSE_HOST 가 같은가 (jp.cloud · cloud · us.cloud)")
    print("  ② pk-lf- 와 sk-lf- 를 서로 바꿔 붙이지 않았는가")
    print("  ③ 키 앞뒤에 공백이나 따옴표가 붙지 않았는가")
    print("  ④ localhost 라면 컨테이너가 떠 있는가 (docker compose ps)")
    sys.exit(1)
print("인증     : 통과")

trace = lf.trace(
    name="trace-hello",
    input=QUESTION,
    tags=["w3d2", "hello"],
    metadata={"host": settings.langfuse_host},
)
generation = trace.generation(
    name="claude",
    model=settings.llm_model,
    input=[{"role": "system", "content": SYSTEM},
           {"role": "user", "content": QUESTION}],
    start_time=datetime.now(timezone.utc),
)

# 클로드 실제 호출
key = settings.anthropic_api_key
try:
    if key is None:
        raise RuntimeError(".env 의 ANTHROPIC_API_KEY 가 비어 있습니다")
    # 클라이언트 생성
    client = anthropic.Anthropic(api_key=key.get_secret_value())
    resp = client.messages.create( 
        model=settings.llm_model,
        max_tokens=settings.max_tokens,
        system=SYSTEM,
        messages=[{"role": "user", "content": QUESTION}],
    )
    answer = "".join(b.text for b in resp.content if b.type == "text")
    generation.end(
        output=answer,
        usage_details={"input": resp.usage.input_tokens,
                       "output": resp.usage.output_tokens},
    )
    trace.update(output=answer)
    print("Claude   :", answer[:60].replace("\n", " "), "...")
    print("토큰     : 입력", resp.usage.input_tokens, "· 출력", resp.usage.output_tokens)
except Exception as e:  
    generation.end(level="ERROR", status_message=f"{type(e).__name__}: {e}")
    trace.update(output=None)
    print("Claude   : 호출 실패 -", type(e).__name__)
    print("           그래도 트레이스는 보냅니다. 화면에서 빨간 ERROR 로 보입니다.")

# 랭퓨즈 내보내기
lf.flush()

# 화면 주소 생성
base = settings.langfuse_host.rstrip("/")
try:
    project_id = lf.client.projects.get().data[0].id  
    print("트레이스 :", f"{base}/project/{project_id}/traces/{trace.id}")
except Exception: 
    print("트레이스 :", trace.id, "(화면 왼쪽 Tracing → Traces 에서 찾으세요)")

보낼 곳 : http://localhost:3000


Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.


인증 실패 : ConnectError
  ① 키를 발급한 리전과 LANGFUSE_HOST 가 같은가 (jp.cloud · cloud · us.cloud)
  ② pk-lf- 와 sk-lf- 를 서로 바꿔 붙이지 않았는가
  ③ 키 앞뒤에 공백이나 따옴표가 붙지 않았는가
  ④ localhost 라면 컨테이너가 떠 있는가 (docker compose ps)


SystemExit: 1

d:\hanwha-agent\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


V4 계열 문법 - 실행은 안할거임 문법만 참고ㅇㅇ

In [7]:
#...
lf = Langfuse(
    public_key=settings.langfuse_public_key,
    secret_key=settings.langfuse_secret_key.get_secret_value(),
    host=settings.langfuse_host,
)

with lf.start_as_current_observation(
    as_type="span", 
    name="trace-hello", 
    input=QUESTION
) as trace:
    with lf.start_as_current_observation(
        as_type="generation", 
        name="claude", 
        model=settings.llm_model, 
        input=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": QUESTION},
        ], 
    ) as generation:
        # Claude 호출 
        generation.update(output=answer)
    trace.update(output=answer)

lf.flush()

AttributeError: 'Langfuse' object has no attribute 'start_as_current_observation'